# v3i YOLOv8n 640 Low-Confidence Validation Review

Inference-only notebook for visual inspection of the full validation split.

Goal: run the trained `YOLOv8n imgsz=640` v3i model over every validation image with low confidence thresholds and save reviewable images/contact sheets.

No training is performed.


## 1. Configuration

In [ ]:
from pathlib import Path

KAGGLE_INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')
WORK_DATASETS_DIR = WORK_ROOT / 'datasets'
OUTPUT_ROOT = WORK_ROOT / 'v3i_yolov8n640_low_conf_val_review'
ANALYSIS_DIR = OUTPUT_ROOT / 'analysis'
WEIGHTS_WORK_DIR = OUTPUT_ROOT / 'weights'

DATASET_FOLDER = 'dataset_yolo_bbox_v3i_li_archaeological_object_merged'
PREBUILT_DATASET_DIR = Path('/kaggle/input/datasets/matanerdy/detection-dataset/dataset_yolo_bbox_v3i_li_archaeological_object_merged')
DATASET_WORK_DIR = WORK_DATASETS_DIR / DATASET_FOLDER
DATASET_YAML = DATASET_WORK_DIR / 'dataset.yaml'
METADATA_PATH = DATASET_WORK_DIR / 'metadata.csv'

# Update these if your Kaggle input uses a different path.
PREFERRED_WEIGHTS_PATHS = [
    Path('/kaggle/input/datasets/matanerdy/detection-dataset/v3i_yolov8n_img640_best.pt'),
    Path('/kaggle/input/datasets/matanerdy/detection-dataset/yolo_v3i_yolov8n_img640_best.pt'),
    Path('/kaggle/input/datasets/matanerdy/detection-dataset/best.pt'),
]

MODEL_TAG = 'v3i_yolov8n_img640'
IMGSZ = 640
NMS_IOU = 0.50
MATCH_IOU = 0.50
COVERAGE_IOU = 0.30
CONF_THRESHOLDS = [0.25, 0.10, 0.05, 0.03, 0.01, 0.005, 0.003, 0.001]
VISUAL_CONFS = [0.05, 0.03, 0.01, 0.005]
MAX_DET = 300
CONTACT_SHEET_COLS = 4
CONTACT_SHEET_ROWS = 4
THUMB_SIZE = (520, 520)

for path in [WORK_DATASETS_DIR, OUTPUT_ROOT, ANALYSIS_DIR, WEIGHTS_WORK_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Dataset folder:', DATASET_FOLDER)
print('Output root:', OUTPUT_ROOT)


## 2. Install Dependencies

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics', 'pandas', 'pillow', 'pyyaml', 'tabulate'],
    check=True,
)


## 3. Copy Dataset to Working Directory

In [ ]:
import shutil
import zipfile

import pandas as pd
import yaml


def find_dataset_dir(folder_name: str) -> Path | None:
    if PREBUILT_DATASET_DIR.exists():
        return PREBUILT_DATASET_DIR
    for dataset_yaml in KAGGLE_INPUT_ROOT.rglob('dataset.yaml'):
        parent = dataset_yaml.parent
        if parent.name == folder_name and (parent / 'images').exists() and (parent / 'labels').exists():
            return parent
    for metadata in KAGGLE_INPUT_ROOT.rglob('metadata.csv'):
        parent = metadata.parent
        if parent.name == folder_name and (parent / 'images').exists() and (parent / 'labels').exists():
            return parent
    return None


def find_dataset_zip(folder_name: str) -> Path | None:
    candidates = sorted(KAGGLE_INPUT_ROOT.rglob(f'{folder_name}.zip'))
    return candidates[0] if candidates else None

source_dataset = find_dataset_dir(DATASET_FOLDER)
if DATASET_WORK_DIR.exists():
    shutil.rmtree(DATASET_WORK_DIR)

if source_dataset is not None:
    print('Copying dataset:', source_dataset)
    shutil.copytree(source_dataset, DATASET_WORK_DIR)
else:
    zip_path = find_dataset_zip(DATASET_FOLDER)
    if zip_path is None:
        raise FileNotFoundError(f'Could not find {DATASET_FOLDER} as a folder or zip under /kaggle/input')
    print('Extracting dataset zip:', zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(WORK_DATASETS_DIR)

if not DATASET_YAML.exists():
    raise FileNotFoundError(f'Missing dataset.yaml: {DATASET_YAML}')

with DATASET_YAML.open('r', encoding='utf-8') as f:
    data_yaml = yaml.safe_load(f)

data_yaml['path'] = str(DATASET_WORK_DIR)
with DATASET_YAML.open('w', encoding='utf-8') as f:
    yaml.safe_dump(data_yaml, f, allow_unicode=True, sort_keys=False)

val_images = sorted((DATASET_WORK_DIR / 'images' / 'val').glob('*'))
val_images = [p for p in val_images if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}]
print('Working dataset:', DATASET_WORK_DIR)
print('Val images:', len(val_images))
print('Dataset yaml:', DATASET_YAML)


## 4. Locate Weights

In [ ]:
import shutil


def find_weights() -> Path:
    for path in PREFERRED_WEIGHTS_PATHS:
        if path.exists():
            return path
    candidates = sorted(KAGGLE_INPUT_ROOT.rglob('*.pt'))
    if not candidates:
        raise FileNotFoundError('Attach trained YOLOv8n 640 v3i best.pt as a Kaggle input, then update PREFERRED_WEIGHTS_PATHS if needed.')
    preferred = [
        p for p in candidates
        if 'v3i' in str(p).lower() and ('yolov8n' in str(p).lower() or 'img640' in str(p).lower() or '640' in str(p).lower())
    ]
    return preferred[0] if preferred else candidates[0]

source_weights = find_weights()
weights_path = WEIGHTS_WORK_DIR / 'v3i_yolov8n_img640_best.pt'
shutil.copy2(source_weights, weights_path)
print('Source weights:', source_weights)
print('Working weights:', weights_path)
print('Size MB:', round(weights_path.stat().st_size / 1024 / 1024, 2))


## 5. Helper Functions

In [ ]:
from __future__ import annotations

from html import escape
import math

import numpy as np
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO

try:
    FONT = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 16)
    SMALL_FONT = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 13)
except Exception:
    FONT = ImageFont.load_default()
    SMALL_FONT = ImageFont.load_default()

GT_COLOR = (0, 210, 70)
TP_COLOR = (0, 170, 255)
FP_COLOR = (255, 120, 0)
TEXT_BG = (0, 0, 0)
TEXT_FG = (255, 255, 255)


def label_path_for_image(image_path: Path) -> Path:
    return DATASET_WORK_DIR / 'labels' / 'val' / f'{image_path.stem}.txt'


def load_yolo_labels(label_path: Path, width: int, height: int) -> list[dict]:
    boxes = []
    if not label_path.exists():
        return boxes
    for line in label_path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        cls, xc, yc, bw, bh = map(float, parts[:5])
        x1 = (xc - bw / 2.0) * width
        y1 = (yc - bh / 2.0) * height
        x2 = (xc + bw / 2.0) * width
        y2 = (yc + bh / 2.0) * height
        boxes.append({
            'class_id': int(cls),
            'box': [max(0, x1), max(0, y1), min(width, x2), min(height, y2)],
            'area': max(0, x2 - x1) * max(0, y2 - y1),
        })
    return boxes


def xyxy_iou(a, b) -> float:
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def greedy_match(gt_boxes: list[dict], pred_boxes: list[dict], iou_thr: float = MATCH_IOU):
    pairs = []
    candidates = []
    for gi, gt in enumerate(gt_boxes):
        for pi, pred in enumerate(pred_boxes):
            candidates.append((xyxy_iou(gt['box'], pred['box']), gi, pi))
    candidates.sort(reverse=True, key=lambda x: x[0])
    used_g, used_p = set(), set()
    for iou, gi, pi in candidates:
        if iou < iou_thr:
            break
        if gi in used_g or pi in used_p:
            continue
        used_g.add(gi)
        used_p.add(pi)
        pairs.append((gi, pi, iou))
    return pairs, used_g, used_p


def draw_text(draw, xy, text, font=SMALL_FONT, pad=3):
    x, y = xy
    bbox = draw.textbbox((x, y), text, font=font)
    draw.rectangle([bbox[0] - pad, bbox[1] - pad, bbox[2] + pad, bbox[3] + pad], fill=TEXT_BG)
    draw.text((x, y), text, fill=TEXT_FG, font=font)


def draw_box(draw, box, color, width=3):
    x1, y1, x2, y2 = box
    for offset in range(width):
        draw.rectangle([x1 - offset, y1 - offset, x2 + offset, y2 + offset], outline=color)


def read_metadata() -> pd.DataFrame:
    if METADATA_PATH.exists():
        return pd.read_csv(METADATA_PATH)
    return pd.DataFrame()

metadata = read_metadata()
meta_by_stem = {}
if not metadata.empty:
    for _, row in metadata.iterrows():
        for col in ['image', 'image_path', 'file_name', 'filename']:
            if col in metadata.columns and pd.notna(row.get(col)):
                meta_by_stem[Path(str(row[col])).stem] = row.to_dict()
                break
print('Metadata rows:', len(metadata))


## 6. Run Low-Confidence Prediction Once

In [ ]:
model = YOLO(str(weights_path))
min_conf = min(CONF_THRESHOLDS)
print('Running prediction with min_conf =', min_conf)

predictions_by_image = {}
for result in model.predict(
    source=[str(p) for p in val_images],
    imgsz=IMGSZ,
    conf=min_conf,
    iou=NMS_IOU,
    max_det=MAX_DET,
    stream=True,
    verbose=False,
):
    image_path = Path(result.path)
    preds = []
    if result.boxes is not None and len(result.boxes) > 0:
        xyxy = result.boxes.xyxy.cpu().numpy()
        confs = result.boxes.conf.cpu().numpy()
        clss = result.boxes.cls.cpu().numpy().astype(int)
        for box, conf, cls in zip(xyxy, confs, clss):
            preds.append({'box': [float(x) for x in box], 'confidence': float(conf), 'class_id': int(cls)})
    predictions_by_image[image_path.name] = preds

print('Predicted images:', len(predictions_by_image))
print('Total predictions at min_conf:', sum(len(v) for v in predictions_by_image.values()))


## 7. Threshold Metrics

In [ ]:
metric_rows = []
per_image_rows = []

for conf in CONF_THRESHOLDS:
    total_tp = total_fp = total_fn = total_gt = covered_gt = total_pred = 0
    for image_path in val_images:
        with Image.open(image_path) as img:
            width, height = img.size
        gt_boxes = load_yolo_labels(label_path_for_image(image_path), width, height)
        pred_boxes = [p for p in predictions_by_image.get(image_path.name, []) if p['confidence'] >= conf]
        pairs, used_g, used_p = greedy_match(gt_boxes, pred_boxes, MATCH_IOU)
        tp = len(pairs)
        fp = len(pred_boxes) - len(used_p)
        fn = len(gt_boxes) - len(used_g)
        image_covered = 0
        for gt in gt_boxes:
            best_iou = max([xyxy_iou(gt['box'], pred['box']) for pred in pred_boxes], default=0.0)
            if best_iou >= COVERAGE_IOU:
                image_covered += 1
        total_tp += tp
        total_fp += fp
        total_fn += fn
        total_gt += len(gt_boxes)
        total_pred += len(pred_boxes)
        covered_gt += image_covered
        meta = meta_by_stem.get(image_path.stem, {})
        per_image_rows.append({
            'conf': conf,
            'image_id': image_path.stem,
            'image_name': image_path.name,
            'region': meta.get('region', ''),
            'n_gt': len(gt_boxes),
            'n_pred': len(pred_boxes),
            'tp': tp,
            'fp': fp,
            'fn': fn,
            'covered_gt_iou_0.30': image_covered,
        })
    precision = total_tp / (total_tp + total_fp) if total_tp + total_fp else 0.0
    recall = total_tp / (total_tp + total_fn) if total_tp + total_fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    metric_rows.append({
        'conf': conf,
        'TP': total_tp,
        'FP': total_fp,
        'FN': total_fn,
        'Precision': precision,
        'Recall': recall,
        'F1': f1,
        'covered_gt_iou_0.30': covered_gt,
        'total_gt': total_gt,
        'coverage_rate_iou_0.30': covered_gt / total_gt if total_gt else 0.0,
        'predictions': total_pred,
        'fp_per_image': total_fp / len(val_images) if val_images else 0.0,
        'predictions_per_image': total_pred / len(val_images) if val_images else 0.0,
    })

threshold_df = pd.DataFrame(metric_rows)
per_image_df = pd.DataFrame(per_image_rows)
threshold_df.to_csv(ANALYSIS_DIR / 'threshold_sweep.csv', index=False)
per_image_df.to_csv(ANALYSIS_DIR / 'per_image_threshold_summary.csv', index=False)
display(threshold_df)
print('Saved:', ANALYSIS_DIR / 'threshold_sweep.csv')


## 8. Draw Individual Review Images

In [ ]:
def annotate_image(image_path: Path, conf: float) -> Image.Image:
    img = Image.open(image_path).convert('RGB')
    width, height = img.size
    gt_boxes = load_yolo_labels(label_path_for_image(image_path), width, height)
    pred_boxes = [p for p in predictions_by_image.get(image_path.name, []) if p['confidence'] >= conf]
    pairs, used_g, used_p = greedy_match(gt_boxes, pred_boxes, MATCH_IOU)

    canvas = img.copy()
    draw = ImageDraw.Draw(canvas)

    for gi, gt in enumerate(gt_boxes):
        color = TP_COLOR if gi in used_g else GT_COLOR
        draw_box(draw, gt['box'], color, width=3)
        x1, y1, *_ = gt['box']
        draw_text(draw, (x1 + 2, max(2, y1 + 2)), 'GT' if gi not in used_g else 'GT matched')

    for pi, pred in enumerate(pred_boxes):
        color = TP_COLOR if pi in used_p else FP_COLOR
        draw_box(draw, pred['box'], color, width=2)
        x1, y1, *_ = pred['box']
        draw_text(draw, (x1 + 2, max(22, y1 + 22)), f"pred {pred['confidence']:.3f}")

    header_h = 74
    out = Image.new('RGB', (width, height + header_h), (245, 245, 245))
    out.paste(canvas, (0, header_h))
    hdraw = ImageDraw.Draw(out)
    meta = meta_by_stem.get(image_path.stem, {})
    title = f"{image_path.name} | conf={conf} | GT={len(gt_boxes)} pred={len(pred_boxes)} TP={len(used_g)} FP={len(pred_boxes)-len(used_p)} FN={len(gt_boxes)-len(used_g)}"
    subtitle = f"region={meta.get('region', '')} | classes={meta.get('source_class_names', meta.get('source_class_name', ''))}"
    hdraw.text((10, 8), title, fill=(0, 0, 0), font=FONT)
    hdraw.text((10, 35), subtitle, fill=(40, 40, 40), font=SMALL_FONT)
    hdraw.text((10, 55), 'green=missed GT, cyan=matched GT/pred, orange=unmatched prediction', fill=(40, 40, 40), font=SMALL_FONT)
    return out

visual_index_rows = []
for conf in VISUAL_CONFS:
    conf_tag = str(conf).replace('.', 'p')
    out_dir = ANALYSIS_DIR / f'visual_conf_{conf_tag}'
    out_dir.mkdir(parents=True, exist_ok=True)
    for image_path in val_images:
        annotated = annotate_image(image_path, conf)
        out_path = out_dir / f'{image_path.stem}_conf_{conf_tag}.jpg'
        annotated.save(out_path, quality=92)
        visual_index_rows.append({'conf': conf, 'image_id': image_path.stem, 'image_name': image_path.name, 'review_image': str(out_path)})

visual_index_df = pd.DataFrame(visual_index_rows)
visual_index_df.to_csv(ANALYSIS_DIR / 'visual_index.csv', index=False)
print('Saved visual images:', len(visual_index_df))
print('Visual root:', ANALYSIS_DIR)


## 9. Contact Sheets and HTML Galleries

In [ ]:
def make_thumbnail(path: Path, size=THUMB_SIZE) -> Image.Image:
    img = Image.open(path).convert('RGB')
    img.thumbnail(size, Image.Resampling.LANCZOS)
    thumb = Image.new('RGB', size, (255, 255, 255))
    x = (size[0] - img.width) // 2
    y = (size[1] - img.height) // 2
    thumb.paste(img, (x, y))
    return thumb

contact_rows = []
for conf in VISUAL_CONFS:
    conf_tag = str(conf).replace('.', 'p')
    img_dir = ANALYSIS_DIR / f'visual_conf_{conf_tag}'
    sheet_dir = ANALYSIS_DIR / f'contact_sheets_conf_{conf_tag}'
    sheet_dir.mkdir(parents=True, exist_ok=True)
    review_images = sorted(img_dir.glob('*.jpg'))
    per_page = CONTACT_SHEET_COLS * CONTACT_SHEET_ROWS
    for page_idx in range(math.ceil(len(review_images) / per_page)):
        batch = review_images[page_idx * per_page:(page_idx + 1) * per_page]
        sheet = Image.new('RGB', (CONTACT_SHEET_COLS * THUMB_SIZE[0], CONTACT_SHEET_ROWS * THUMB_SIZE[1]), (240, 240, 240))
        for idx, path in enumerate(batch):
            thumb = make_thumbnail(path)
            x = (idx % CONTACT_SHEET_COLS) * THUMB_SIZE[0]
            y = (idx // CONTACT_SHEET_COLS) * THUMB_SIZE[1]
            sheet.paste(thumb, (x, y))
        out_path = sheet_dir / f'val_review_conf_{conf_tag}_page_{page_idx + 1:02d}.jpg'
        sheet.save(out_path, quality=92)
        contact_rows.append({'conf': conf, 'page': page_idx + 1, 'contact_sheet': str(out_path)})

contact_df = pd.DataFrame(contact_rows)
contact_df.to_csv(ANALYSIS_DIR / 'contact_sheets.csv', index=False)
display(contact_df)

for conf in VISUAL_CONFS:
    conf_tag = str(conf).replace('.', 'p')
    rows = visual_index_df[visual_index_df['conf'] == conf].copy()
    stats = per_image_df[per_image_df['conf'] == conf].set_index('image_id')
    cards = []
    for _, row in rows.iterrows():
        image_id = row['image_id']
        stat = stats.loc[image_id].to_dict() if image_id in stats.index else {}
        rel = Path(row['review_image']).relative_to(ANALYSIS_DIR)
        cards.append(
            '<div class="card">'
            f'<a href="{escape(str(rel))}" target="_blank"><img src="{escape(str(rel))}"></a>'
            f'<div><b>{escape(row["image_name"])}</b></div>'
            f'<div>region: {escape(str(stat.get("region", "")))}</div>'
            f'<div>GT={stat.get("n_gt", "")} pred={stat.get("n_pred", "")} TP={stat.get("tp", "")} FP={stat.get("fp", "")} FN={stat.get("fn", "")}</div>'
            '</div>'
        )
    html = """<!doctype html>
<html><head><meta charset="utf-8"><title>v3i low-conf review CONF_VALUE</title>
<style>
body { font-family: Arial, sans-serif; margin: 20px; background: #f7f7f7; }
.grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(360px, 1fr)); gap: 16px; }
.card { background: white; border: 1px solid #ddd; padding: 10px; }
.card img { width: 100%; height: auto; display: block; }
.legend { margin-bottom: 16px; }
</style></head><body>
<h1>v3i YOLOv8n 640 low-confidence validation review</h1>
<div class="legend">conf=CONF_VALUE; green=missed GT, cyan=matched GT/pred, orange=unmatched prediction.</div>
<div class="grid">CARDS_VALUE</div>
</body></html>""".replace('CONF_VALUE', str(conf)).replace('CARDS_VALUE', ''.join(cards))
    html_path = ANALYSIS_DIR / f'gallery_conf_{conf_tag}.html'
    html_path.write_text(html, encoding='utf-8')
    print('Saved HTML gallery:', html_path)


## 10. Archive Outputs

In [ ]:
import shutil

summary_path = ANALYSIS_DIR / 'summary.md'
with summary_path.open('w', encoding='utf-8') as f:
    f.write('# v3i YOLOv8n 640 Low-Confidence Validation Review\n\n')
    f.write(f'- dataset: `{DATASET_FOLDER}`\n')
    f.write(f'- weights: `{source_weights}`\n')
    f.write(f'- imgsz: `{IMGSZ}`\n')
    f.write(f'- NMS IoU: `{NMS_IOU}`\n')
    f.write(f'- validation images: `{len(val_images)}`\n\n')
    f.write('## Threshold Sweep\n\n')
    f.write(threshold_df.to_markdown(index=False))
    f.write('\n\n## Visual Outputs\n\n')
    for conf in VISUAL_CONFS:
        conf_tag = str(conf).replace('.', 'p')
        f.write(f'- `gallery_conf_{conf_tag}.html`\n')
        f.write(f'- `visual_conf_{conf_tag}/`\n')
        f.write(f'- `contact_sheets_conf_{conf_tag}/`\n')

archive_base = WORK_ROOT / 'v3i_yolov8n640_low_conf_val_review_outputs'
archive_path = shutil.make_archive(str(archive_base), 'zip', OUTPUT_ROOT)
print('Summary:', summary_path)
print('Archive:', archive_path)
print('Open HTML galleries under:', ANALYSIS_DIR)
